# reader

> parsing lisp text into an AST

### nbdev prologue

In [ ]:
#| default_exp reader

### Atoms and Literals

In [ ]:
#| export
import ast, re
from fastcore.basics import basic_repr, store_attr, first, last
from compact.types import *

help understand literals in Lisp

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

In Lisp, a **literal** is a value written directly in the program.

Examples:

```lisp
42
"hello"
#t
```

These usually evaluate to themselves:

```lisp
42        ; evaluates to the number 42
"hello"   ; evaluates to the string "hello"
#t        ; evaluates to true
```

The interesting contrast is with **symbols**:

```lisp
x
```

`x` is not usually the literal string `"x"`. It means: “look up the value bound to the name `x`.”

So for our Python implementation, we can often represent Lisp literals using ordinary Python values:

```python
42, "hello", True
```

But symbols need a separate representation, because `"x"` as a Lisp string and `x` as a Lisp variable name are different things.

<details class='token-usage-details' markdown='1'><summary>$0.0048</summary>

`total=31,230 | in=31,049 | out=181 | cached=96.9% | $0.0048 | gpt-5.5`

</details>


In [ ]:
ast.literal_eval("123"), ast.literal_eval('"hello world"'), ast.literal_eval("2+3j")

(123, 'hello world', (2+3j))

In [ ]:
#| export
def Atom(s):
    "parse atoms, default this is a symbol"
    if s == "#t": return True
    if s == "#f": return False
    try: return ast.literal_eval(s)
    except Exception: return Symbol(s)


In [ ]:
# class Symbol:
#     "symbol is a `name` in Lisp"
#     def __init__(self, s): store_attr()
#     def __str__(self): return f"{self.s}"
# 
#     def __eq__(self, other): return isinstance(other, Symbol) and self.s == other.s
#     def __hash__(self): return hash(self.s)
#     
#     __repr__ = basic_repr()

In [ ]:
Atom("123"), Atom('"hello world"'), Atom("2+3j"), Atom("lambda")

(123, 'hello world', (2+3j), Symbol(s='lambda'))

In [ ]:
Atom("+") == Atom("+")

True

In [ ]:
Atom("+") == Atom('"+"')

False

### Reader: Lexer and Parser
parse strings into lisp expressions

In [ ]:
#| export
STR = r'"(?:\\.|[^"\\])*"'  # string with spaces etc.
COMMENT = r';[^\n]*'

DOT = r'\.'

ATOM = r'''[^\s()`',;]+'''

UNQ_SPL = ',@'              # unquote splice
PAREN = '[()]'
QUOTE = "[`',]"

# order matters `,@` needs to show up before the single character `,`
TOKEN_RE = "|".join([STR, COMMENT, DOT, UNQ_SPL, PAREN, QUOTE, ATOM])

def lexer(s): return [t for t in re.findall(TOKEN_RE, s) if not t.startswith(';')]

In [ ]:
lexer("( + 1 2 )")

['(', '+', '1', '2', ')']

In [ ]:
lexer("'( + 1 2 )")

["'", '(', '+', '1', '2', ')']

In [ ]:
lexer("`( + 1 2 )")

['`', '(', '+', '1', '2', ')']

In [ ]:
lexer('`(+ ,@xs 2)')

['`', '(', '+', ',@', 'xs', '2', ')']

In [ ]:
lexer("(+ 1 2)")
# expect: ["(", "+", "1", "2", ")"]

['(', '+', '1', '2', ')']

In [ ]:
lexer("'(1 2 3)")
# expect: ["'", "(", "1", "2", "3", ")"]

["'", '(', '1', '2', '3', ')']

In [ ]:
lexer("'(10 . 20)")
# expect: ["'", "(", "10", ".", "20", ")"]

["'", '(', '10', '.', '20', ')']

In [ ]:
lexer("(1 . 2)")
# expect: ["(", "1", ".", "2", ")"]

['(', '1', '.', '2', ')']

In [ ]:
#| export
def parse_list(sl):
    match sl:
        case [*head, Symbol(s="."), tail] if head: return list2pair(head, tail)
        case _ if Symbol(s=".") in sl: raise SyntaxError("malformed dotted list")
        # case _ if any(isinstance(o, Symbol) and o.s == "." for o in sl): raise SyntaxError("malformed dotted list")
        # case [*_, Symbol(s="."), *_]: raise SyntaxError("malformed dotted list")
        case _: return sl

In [ ]:
#| export
def parser(toks):
    if not toks: raise SyntaxError("malformed list: unexpected EOF")
    

    t = toks.pop(0)
    if t == ')': raise SyntaxError("malformed list: unexpected )")
    if t == '(':
        sl = []
        while toks and toks[0] != ')': sl.append(parser(toks))
        if not toks: raise SyntaxError("malformed list: missing )")
        toks.pop(0)
        return parse_list(sl)
    sugar = {
        "'": "quote",
        "`": "quasiquote",
        ",": "unquote",
        ",@": "unquote-splicing",
    }
    if t in sugar: return [Symbol(sugar[t]), parser(toks)]

    return Atom(t)

In [ ]:
parser(lexer('`(+ ,@xs 2)'))

[Symbol(s='quasiquote'),
 [Symbol(s='+'), [Symbol(s='unquote-splicing'), Symbol(s='xs')], 2]]

In [ ]:
#| export
def parse(s): 
    toks = lexer(s)
    r = parser(toks)
    if not toks: return r
    raise SyntaxError("unexpected tokens after expression")

In [ ]:
parse('`(+ ,@xs 2)')

[Symbol(s='quasiquote'),
 [Symbol(s='+'), [Symbol(s='unquote-splicing'), Symbol(s='xs')], 2]]

In [ ]:
parse('`(+ ,@xs 2)')

[Symbol(s='quasiquote'),
 [Symbol(s='+'), [Symbol(s='unquote-splicing'), Symbol(s='xs')], 2]]

In [ ]:
parse("'(1 . 2)")

[Symbol(s='quote'), compact.types.Pair(car=1, cdr=2)]

In [ ]:
parse("'(1  2 . 3)")

[Symbol(s='quote'),
 compact.types.Pair(car=1, cdr=compact.types.Pair(car=2, cdr=3))]

In [ ]:
try: parse("'(. 2)")
except SyntaxError: pass

In [ ]:
try: parse("'(1 . 2 3)")
except SyntaxError: pass

In [ ]:
#| export
def parse_all(s):
    toks = lexer(s)
    exprs = []
    while toks: exprs.append(parser(toks))
    return exprs

In [ ]:
parse_all("""
(define x 10)
(+ x 5)
""")

[[Symbol(s='define'), Symbol(s='x'), 10], [Symbol(s='+'), Symbol(s='x'), 5]]

### nbdev postscript

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()